### Setup

In [1]:
import matplotlib.pyplot as plt
import scipy as sc
from scipy.stats import pearsonr
import numpy as np
import pandas as pd
import seaborn as sns
import sys
import os
import torch
from tqdm import tqdm
import scipy.optimize as opt
import subprocess

In [2]:
sys.path.insert(0, os.getcwd())

from model import GPTConfig, GPT
from lib.utils import get_batch, get_ckpt_model, two_phase_opt

In [3]:
dataset='shakespeare_char'

## NanoGPT Experiments

### Experiment 1: MHA LRB

In [5]:
n_head_choices = [1,2,3,4,5,6,7,8,16,32,64]
n_samples = 350
d=128
ckpt_dir="experiments"

In [6]:
%%script false --no-raise-error
for n_head in [64]:
    head_size = d//n_head
    train_cmd = [
        "python", "train.py", "config/train_shakespeare_char.py",
        f"--head_size={head_size}",
        f"--n_head={n_head}",
        f"--out_dir={ckpt_dir}",
        f"--out_fname=ckpt_h{n_head}.pt",
        f"--device=xpu",
    ]
    print(' '.join(train_cmd))
    subprocess.run(train_cmd, stdout=subprocess.DEVNULL)

In [7]:
%%script false --no-raise-error
rows = []
for n_head in n_head_choices:
    ckpt_fname = f"ckpt_h{n_head}.pt"
    model = get_ckpt_model(ckpt_fname, ckpt_dir=ckpt_dir)
    batch_size = model.config.batch_size
    n = model.config.block_size

    print(f"Evaluating {ckpt_fname}")
    
    losses = np.zeros(n_samples)
    for i in tqdm(range(n_samples)):
        X, Y = get_batch('eval', os.path.join('data', dataset), 'cpu', n, batch_size)
        logits, loss = model(X, Y)
        losses[i]= loss
            
    rows.append({
        'head_size': model.config.head_size,
        'n_head': n_head,
        'perplexity': np.exp(np.mean(losses)),
    })
    
df1 = pd.DataFrame(rows)
df1.set_index('n_head', inplace=True)
df1.to_csv("experiments/ex1_results.csv")

### Experiment 2: FMHA LRB

In [8]:
n_head_choices = [1,2,3,4,5,6,7,8]
n_samples = 350
d=128
ckpt_dir="experiments"

In [9]:
%%script false --no-raise-error
for n_head in [4,5,6,7,8]:
    head_size = 40 # > n=32
    train_cmd = [
        "python", "train.py", "config/train_shakespeare_char.py",
        f"--head_size={head_size}",
        f"--n_head={n_head}",
        f"--out_dir={ckpt_dir}",
        f"--out_fname=ckpt_fmha_h{n_head}.pt",
        f"--device=xpu",
    ]
    print(' '.join(train_cmd))
    subprocess.run(train_cmd, stdout=subprocess.DEVNULL)

In [10]:
%%script false --no-raise-error
rows = []
for n_head in n_head_choices:
    ckpt_fname = f"ckpt_fmha_h{n_head}.pt"
    model = get_ckpt_model(ckpt_fname, ckpt_dir=ckpt_dir)
    assert(model.config.n_embd == d)
    
    print(f"Evaluating {ckpt_fname}")
    
    losses = np.zeros(n_samples)
    for i in tqdm(range(n_samples)):
        X, Y = get_batch('eval', os.path.join('data', dataset), 'cpu', n, batch_size)
        logits, loss = model(X, Y)
        losses[i]= loss

    rows.append({
        'n_head': n_head,
        'perplexity': np.exp(np.mean(losses)),
    })
df2 = pd.DataFrame(rows)
df2.to_csv("experiments/ex2_results.csv")

### Experiment 3: Equivalent Non-Identifiable Distribution for MHA

In [11]:
n_head_choices = [1,2,3,4,5,6,7,8,16,32,64]
n_samples = 100
d=128
ckpt_dir="experiments"

In [12]:
%%script false --no-raise-error
rows = []
for n_head in n_head_choices:
    ckpt_fname = f"ckpt_h{n_head}.pt"
    model = get_ckpt_model(ckpt_fname, ckpt_dir=ckpt_dir)
    assert(model.config.n_embd == d)
    h = model.config.n_head
    n = model.config.block_size
    hs = model.config.head_size
    batch_size = model.config.batch_size

    print(f"Evaluating {ckpt_fname}")
    
    opt_vals = []
    losses = np.zeros(n_samples)
    if hs < n :
        for i in tqdm(range(n_samples)):
            X, Y = get_batch('eval', os.path.join('data', dataset), 'cpu', n, batch_size)
            A,T,v = model.get_matricies(X,0)
            res = two_phase_opt(n,A,T,X.numpy(),col_idx=0)
            if(res != None):
                opt_vals.append(res[0])

        if(len(opt_vals) == 0):
            min_opt_val = 0
        else:
            min_opt_val = np.min(np.array(opt_vals))

        rows.append({
            'head_size': d//n_head,
            'n_head': n_head,
            'perplexity': np.exp(np.mean(losses)),
            'min_opt_val': min_opt_val,
        })
    
df3 = pd.DataFrame(rows)
df3['log_min_opt_val'] = np.log(df3['min_opt_val'])
df3.to_csv("experiments/ex3_results.csv")

### Experiment 4: Removing the Low Rank Bottleneck

In [15]:
n_samples = 100
model = get_ckpt_model("ckpt_h64.pt", ckpt_dir="experiments")
n = model.config.block_size
batch_size = model.config.batch_size

In [17]:
results = np.zeros(n_samples)

for j in tqdm(range(n_samples)):
    opt_vals = np.zeros(n)
    losses = np.zeros(n_samples)
    X, _ = get_batch('eval', os.path.join('data', dataset), 'cpu', n, batch_size)
    A,T,v = model.get_matricies(X,0)
    for i in range(n):
        res = two_phase_opt(n,A,T,X.numpy(),col_idx=i)
        if(res != None):
            opt_val, a = res
            opt_vals[i] = opt_val
        else:
            opt_vals[i] = -1
            
    opt_vals = np.array(opt_vals)
    results[j] = np.max(opt_vals)

  0%|          | 0/100 [00:00<?, ?it/s]

column 0: z*=0.007683843323793349
column 1: z*=0.0009906442285223527
column 2: z*=3.1097373174970855e-05
column 3: z*=6.534366469512776e-05
column 4: z*=0.0007518841034803714
column 5: z*=0.0037630817654772386
column 6: z*=0.0001312587728321546
column 7: z*=0.00012122371806189136
column 8: z*=0.0015146951871267689
column 9: z*=0.008262888909674082
column 10: z*=0.0005135276141663425
column 11: z*=0.000675534594293481
column 12: z*=0.014349767341606423
column 13: z*=0.0012181563013617946
column 14: z*=0.016740941660964737
column 15: z*=0.0012604267002664926
column 16: z*=0.020452495436858786
column 17: z*=0.5144085633432249
column 18: z*=1.283711939608719
column 19: z*=3.0122080561880433
column 20: z*=0.30400267909867384


  1%|          | 1/100 [00:56<1:32:52, 56.29s/it]

column 21: z*=1.9296353420544534
column 22: z*=5.898904281784917
column 23: z*=3.668152040951686
column 24: z*=2.9407351614360553
column 25: z*=3.44659539744172
column 26: z*=8.388219963148305
column 27: z*=71.35414996963222
column 28: z*=49.66678997360315
column 29: z*=97.59352368208313
column 30: z*=97.59352368208313
column 31: z*=97.59352368208313
column 0: z*=0.22079982538890386
column 1: z*=0.00016936005658951814
column 2: z*=0.16231446150895645
column 3: z*=0.0028675810861650067
column 4: z*=0.01910453097314413
column 5: z*=0.0005961775408640513
column 6: z*=0.06340977502805556
column 7: z*=0.7549031634767953
column 8: z*=2.6918211616918386
column 9: z*=1.6238177790238781
column 10: z*=1.17160481267931
column 11: z*=0.30498335400260856
column 12: z*=2.2880240043175046
column 13: z*=1.6491158283460174
column 14: z*=2.4663984764513858
column 15: z*=3.676711484454344
column 16: z*=4.259305095370587
column 17: z*=5.065635423017878
column 18: z*=5.007276672690628
column 19: z*=5.79669

  2%|▏         | 2/100 [01:39<1:19:49, 48.87s/it]

...

column 26: z*=9.455811014708228
column 27: z*=11.859839393754854
column 28: z*=18.287370106718246
column 29: z*=15.919766715271988
column 30: z*=23.381587297136033
column 31: z*=60.88091547980319
column 0: z*=0.0001210060872136598
column 1: z*=7.155314361506626e-05
column 2: z*=0.007674599256016101
column 3: z*=2.029143554338475e-05
column 4: z*=0.0003131649624954491
column 5: z*=0.00011745431775622706
column 6: z*=0.00012302047203327504
column 7: z*=7.537954072844855e-05
column 8: z*=0.0032517305971717647
column 9: z*=0.0016791500311788
column 10: z*=0.060625476529069466
column 11: z*=0.06703127253710228
column 12: z*=1.2607935196636966
column 13: z*=1.902755261334673
column 14: z*=4.001185009333861
column 15: z*=4.1126826448760045
column 16: z*=6.006896964098047
column 17: z*=6.105331930286695
column 18: z*=6.326379322047526
column 19: z*=5.7907629756044
column 20: z*=7.144923970510373
column 21: z*=8.211212298825803
column 22: z*=3.178252598814148


 10%|█         | 10/100 [07:52<1:09:10, 46.11s/it]

column 23: z*=4.102482746889468
column 24: z*=11.429991375510028
column 25: z*=12.155284817575888
column 26: z*=74.10200285534842
column 27: z*=103.39140301032072
column 28: z*=91.64092214110133
column 29: z*=162.93379640059547
column 30: z*=54.55347609225994
column 31: z*=54.55347609225994
column 0: z*=0.001000259843122242
column 1: z*=0.0023312394593615338
column 2: z*=5.2987014587557724e-05
column 3: z*=0.0007515077438426943
column 4: z*=0.00013355862877428848
column 5: z*=5.642902049118881e-05
column 6: z*=0.00018260203581643708
column 7: z*=0.00020853311147295273
column 8: z*=0.018724133981895985
column 9: z*=0.0044512150083586555
column 10: z*=0.000252339759969189


 10%|█         | 10/100 [08:44<1:18:42, 52.47s/it]


KeyboardInterrupt: 

In [45]:
np.min(np.array(results))

np.float64(340.70335254130237)

### Results

In [37]:
%%script false --no-raise-error
fig, ax = plt.subplots(1,1,figsize=(6,4))
sns.lineplot(df1, x='n_head', y='perplexity', marker='s', ax=ax, label="MHA")
sns.lineplot(df2, x='n_head', y='perplexity', marker='s', ax=ax, label="FMHA")
ax.set_xlabel(r"$h$")
ax.set_ylabel("perplexity")
ax.set_title("Low Rank Bottleneck (MHA)")
# plt.savefig("images/lrb_mha_fhma.png")
plt.close()

In [38]:
%%script false --no-raise-error
fig,ax = plt.subplots(1,1,figsize=(6,4))
sns.lineplot(df, x='n_head', y='log_min_opt_val', ax=ax, marker='s')
ax.set_title("LRB Influence in MHA")
ax.set_ylabel(r"$\log(z^*)$")
ax.set_xlabel(r"$h$")
# plt.savefig("images/lrb_opt_mha.png")
plt.close()